In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

RUN = Path("results/runs/latest")          # the symlink you set up

solves = pd.read_csv(RUN / "solves.csv")
joints = pd.read_csv(RUN / "joints.csv")
constraints = pd.read_csv(RUN / "constraints.csv")
timings = pd.read_csv(RUN / "timings.csv")

print(f"{len(solves)} solve rows | solvers: {sorted(solves.solver_name.unique())}")
print(f"tols: {sorted(solves.orientation_tol.unique())} | seeds: {sorted(solves.seed.unique())}")
print(f"methods: {sorted(solves.generation_method.unique())}")
solves.head()

18000 solve rows | solvers: ['Ipopt', 'NLopt', 'SNOPT']
tols: [np.float64(0.0), np.float64(0.0001), np.float64(0.001), np.float64(0.01)] | seeds: [np.int64(2), np.int64(9), np.int64(42)]
methods: ['joint_space']


,solve_id,scenario_id,seed,orientation_tol,solver_name,generation_method,success,solution_result,optimal_cost,evaluated_cost,feasible,max_violation,common_stationarity_inf,common_stationarity_2,native_optimality,time_median,n_repeats,timing_matches_record,error_type,error_message
0,2_0.0_0_SNOPT,0,2,0.0,SNOPT,joint_space,True,SolutionResult.kSolutionFound,13.259804,13.259804,True,2.831189e-10,2.664535e-15,2.878027e-15,7.000000,0.002633,10,True,NaN,NaN
1,2_0.0_0_Ipopt,0,2,0.0,Ipopt,joint_space,False,SolutionResult.kInfeasibleConstraints,18.109689,18.109689,True,5.109057e-05,6.108660e+00,8.511096e+00,6.108660,0.021492,10,True,NaN,NaN
2,2_0.0_0_NLopt,0,2,0.0,NLopt,joint_space,False,SolutionResult.kSolverSpecificError,NaN,NaN,False,NaN,NaN,NaN,NaN,0.002297,10,True,NonFiniteSolution,7 non-finite entries
3,2_0.0_1_SNOPT,1,2,0.0,SNOPT,joint_space,False,SolutionResult.kSolverSpecificError,3.854855,3.854855,False,7.378587e-01,1.998401e-15,2.186886e-15,3.300000,0.003794,10,True,NaN,NaN
4,2_0.0_1_Ipopt,1,2,0.0,Ipopt,joint_space,True,SolutionResult.kSolutionFound,12.956898,12.956898,True,9.996749e-09,4.983655e-03,7.284929e-03,0.005006,0.035874,10,True,NaN,NaN


In [2]:
(solves
 .groupby(["orientation_tol", "solver_name"])
 .agg(n=("success", "size"),
      success_rate=("success", "mean"),
      feasible_rate=("feasible", "mean"),
      nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()))
 .round(3))

n  success_rate  feasible_rate  nan_rate
orientation_tol solver_name                                             
0.0000          Ipopt        1500         0.490          0.631     0.000
                NLopt        1500         0.448          0.448     0.549
                SNOPT        1500         0.581          0.591     0.000
0.0001          Ipopt        1500         0.649          0.663     0.000
                NLopt        1500         0.455          0.455     0.544
                SNOPT        1500         0.577          0.588     0.000
0.0010          Ipopt        1500         0.673          0.685     0.000
                NLopt        1500         0.451          0.451     0.546
                SNOPT        1500         0.581          0.589     0.000
0.0100          Ipopt        1500         0.690          0.691     0.000
                NLopt        1500         0.468          0.468     0.529
                SNOPT        1500         0.585          0.589     0.000

In [ ]:
(feas_scens
 .groupby(["solver_name"])
 .agg(n=("success", "size"),
      success_rate=("success", "mean"),
      feasible_rate=("feasible", "mean"),
      nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()))
 .round(3))

In [21]:
fscens = solves[solves.feasible].scenario_id.unique()
n_scens = len(solves.scenario_id.unique())

feas_scens = solves[solves["scenario_id" ].isin(fscens)]

print(f"total scenarios: {n_scens}")
print(f"scenarios (>=1 feasible soln found): {len(fscens)} ({round(len(fscens)/n_scens, 4)*100}%)")
print(f"solve rows (only feasible scenarios): {len(feas_scens)}")

(feas_scens
 .groupby(["solver_name"])
 .agg(n=("success", "size"),
      success_rate=("success", "mean"),
      feasible_rate=("feasible", "mean"),
      nan_rate=("error_type", lambda s: (s == "NonFiniteSolution").mean()))
 .round(3))


total scenarios: 6000
scenarios (>=1 feasible soln found): 4736 (78.93%)
solve rows (only feasible scenarios): 14208


,n,success_rate,feasible_rate,nan_rate
solver_name,,,,
Ipopt,4736,0.793,0.845,0.000
NLopt,4736,0.577,0.577,0.422
SNOPT,4736,0.736,0.746,0.000


# timing

In [ ]:
# median time - total by solver
(solves[solves.feasible]                      # only time solves that actually worked
 .groupby(["solver_name"])["time_median"]
 .median()
 .mul(1000).round(2)                          # ms
)

solver_name
Ipopt    16.01
NLopt     2.35
SNOPT     3.24
Name: time_median, dtype: float64

In [ ]:
# median solve time - by solver x orientation
(solves[solves.feasible]                      # only time solves that actually worked
 .groupby(["orientation_tol", "solver_name"])["time_median"]
 .median()
 .mul(1000).round(2)                          # ms
 .unstack("solver_name"))

solver_name,Ipopt,NLopt,SNOPT
orientation_tol,,,
0.0000,47.51,3.08,4.63
0.0001,18.61,2.42,3.88
0.0010,13.29,2.01,2.80
0.0100,8.58,1.69,2.35


In [ ]:
# pivot so each scenario has both solvers side by side
piv = solves.pivot_table(
    index=["seed", "orientation_tol", "scenario_id"],
    columns="solver_name",
    values=["feasible", "common_stationarity_inf", "evaluated_cost"],
    aggfunc="first")

both_feas = piv[("feasible", "SNOPT")].astype(bool) & piv[("feasible", "Ipopt")].astype(bool)
comp = piv[both_feas]
print(f"{both_feas.sum()} scenarios where both SNOPT and IPOPT feasible")

# stationarity residuals side by side, where both feasible
comp[[("common_stationarity_inf", "SNOPT"), ("common_stationarity_inf", "Ipopt")]].describe()

In [ ]:
cost_snopt = comp[("evaluated_cost", "SNOPT")]
cost_ipopt = comp[("evaluated_cost", "Ipopt")]
rel_gap = ((cost_snopt - cost_ipopt).abs() / cost_ipopt.abs())
print("relative cost gap SNOPT vs IPOPT (both feasible):")
print(rel_gap.describe())